# 大模型的外推性
外推性是指大模型在训练时和预测是的输入长度不一致，导致模型的泛化能力下降的问题。例如，如果一个模型在训练时值使用了512个token, 那么在预测时如果使用了1024个token, 那么模型的泛化能力就会下降。

# Rotary Position Embedding，RoPE <br>
定义一个长度为N的输入序列：
$$\mathbb{S}_N = \{w_i\}_{i=1}^{N}$$
其中$w_i$代表输入序列中第i个token,而输入序列$\mathbb{N}$对应的embedding表示为：
$$\mathbb{E}_N = \{x_i\}_{i=1}^{N}$$
其中$x_i$代表输入序列$\mathbb{N}$中第i个token的d维词嵌入向量。<br>
在做self-attention前，会用词嵌入向量计算q,k,v向量同时加入位置信息，函数表达式如下：
$$\begin{aligned}
q_m = f_q(x_m, m) \\
k_n = f_k(x_n, n) \\
v_n = f_v(x_n, n) \\
\end{aligned}$$
其中，$q_m$表示第m个token对应的词向量$x_m$集成位置信息m之后的query向量，$k_n$表示第n个token对应的词向量$x_n$集成位置信息n之后的key向量，$v_n$表示第n个token对应的词向量$x_n$集成位置信息n之后的value向量。<br>
而基于transformer的位置编码方法都是着重于构造一个合适的$f(q,k,v)$函数形式，使得在计算self-attention时，$q_m$和$k_n$的相似度能够反映出$m$和$n$之间的相对位置关系。<br>

RoPE论文提出假定query向量$q_m$和key向量$k_n$之间的内积操作可以被一个函数$g$表示，即：
$$
<f_q(x_m, m), f_k(x_n, b)> = g(x_m, x_n, m-n)
$$
接下来的目标就是寻找一个合适的函数$g$，使得上述等式成立。<br>
RoPE论文中，作者提出了一个基于旋转的函数$g$，即：
$$
f_q(x_m, m) = (W_q*x_m)e^{i*m*\theta} \\
f_k(x_n, n) = (W_k*x_n)e^{i*n*\theta} \\
g(x_m, x_n, m-n) = Re[(W_q*x_m)e^{i*m*\theta} \cdot (W_k*x_n)e^{i*n*\theta}] = Re[(W_q*x_m)(W_k*x_n) \cdot e^{i(m-n)\theta}]
$$
这里的Re表示复数的实部，进一步$f_q$可以表示成下面的式子：
$$
f_q(x_m, m) =
\begin{bmatrix}
\cos(\theta m) & -\sin(\theta m) \\
\sin(\theta m) & \cos(\theta m)
\end{bmatrix}
\begin{bmatrix}
W_{q,1}x_{m,1} & W_{q,2}x_{m,2} \\
W_{q,3}x_{m,3} & W_{q,4}x_{m,4}
\end{bmatrix}
\begin{bmatrix}
x_{m,1} \\
x_{m,2}
\end{bmatrix}
$$

上述公式转化由欧拉公式$e^{ix} = \cos x + i\sin x$推导而来。最终公式：
$$
f_q(x_m, m) = 
\begin{bmatrix}
\cos(\theta m) & -\sin(\theta m) \\
\sin(\theta m) & \cos(\theta m)
\end{bmatrix}
\begin{bmatrix}
q_{m,1} \\
q_{m,2}
\end{bmatrix}
$$

将上述公式带入到$g(x_m, x_n, m-n)$中，得到：
$$
g(x_m, x_n, m-n) = y_m^T \, R_{m-n} \, y_n = (W_q x_m)^T
\begin{bmatrix}
\cos(\theta (m-n)) & -\sin(\theta (m-n)) \\
\sin(\theta (m-n)) & \cos(\theta (m-n))
\end{bmatrix} (W_k x_n)
$$

关于这里的$\theta$取值：（i表示一个token的嵌入表示中的第i个维度，m表示第m个token）
$$
\theta_i = 10000^{-2(i-1)/d}, \quad i = 1, \dots, d/2
$$

$$
R_i(m) =
\begin{bmatrix}
\cos(\theta_i m) & -\sin(\theta_i m) \\
\sin(\theta_i m) & \cos(\theta_i m)
\end{bmatrix}
$$


In [4]:
import torch
import torch.nn as nn
import sys
import os
from typing import  Optional
sys.path.append("/home/jx/experiment/BedRock-llm") # 修改为本地路径
from model import BedrockV3Config
from collections.abc import Callable
from transformers.utils.generic import maybe_autocast




In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self, config: BedrockV3Config, device=None):
        super().__init__()

        self.max_seq_len = config.max_position_embeddings
        self.original_max_seq_len = config.max_position_embeddings

        self.config = config
        # 随着rope的发展，出现了多种变体，这里采用原始论文中不加scaling的版本
        inv_freq = self.compute_default_rope_parameters(config, device)

        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.register_buffer("original_inv_freq", inv_freq.clone(), persistent=False)

    @staticmethod
    def compute_default_rope_parameters(
        config: Optional[BedrockV3Config] = None,  # BedrockV3Config类型的可选参数，用于配置模型参数
        device: Optional["torch.device"] = None,  # torch.device类型的可选参数，指定计算设备
        seq_len: Optional[int] = None  # 整数类型的可选参数，指定序列长度
    ):
        '''
        计算ROPE中的theta参数，即旋转角度中用到的频率
        '''
    
        base = config.rope_parameters['rope_theta']
        dim = getattr(config, "head_dim", None) or config.hidden_size // config.num_attention_heads
        # Compute the inverse frequencies
        inv_freq = 1.0 / (
            base ** (torch.arange(0, dim, 2, dtype=torch.int64).to(device=device, dtype=torch.float64) / dim)
        ) # [dim/2]
        return inv_freq

    @torch.no_gard()
    def forward(self, x, position_ids):
        '''
        依据频率和位置信息计算旋转角度并得到旋转角度的三角函数值
        '''
        # self.inv_freq[None, :, None]->[1,dim/2,1]
        inv_freq_expanded = self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1).to(x.device) # [batch,dim/2,1]
        position_ids_expanded = position_ids[:, None, :].float() # [batch,1,seq_len]

        device_type = x.device.type if isinstance(x.device, str) and x.device.type != "mps" else "cpu"
        with maybe_autocast(device_type=device_type, enabled=False): # force float32
            # 依据频率和位置信息计算旋转角度
            freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1,2) # [batch,dim/2,1] @ [batch,1,seq_len] -> [batch,dim/2,seq_len]-> [batch,seq_len,dim/2]
            emb = torch.cat((freqs, freqs), dim=-1) # [batch,seq_len,dim] 将dim分成前后两部分，第一个维度和第dim/2+1个维度组合进行旋转
            cos = emb.cos() # [batch,seq_len,dim]
            sin = emb.sin() # [batch,seq_len,dim]

        return cos.to(dtype=x.dtype), sin.to(dtype=x.dtype)


def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2] # 前一半维度
    x2 = x[..., x.shape[-1] // 2 :] # 后一半维度
    return torch.cat((-x2, x1), dim=-1) # 将后一半维度取负并与前一半维度调换位置拼接

def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    cos = cos.unsqueeze(unsqueeze_dim) # [batch,1,seq_len,dim]
    sin = sin.unsqueeze(unsqueeze_dim) # [batch,1,seq_len,dim]
    q_embed = (q * cos) + (rotate_half(q) * sin) # 这里q*cos，在q的dim维度上全部乘以cos, rotate_half(q) * sin,在q的dim维度上全部乘以sin,然后相加对应公式f_q(x_m, m)
    k_embed = (k * cos) + (rotate_half(k) * sin) # 这里k*cos，在k的dim维度上全部乘以cos, rotate_half(k) * sin,在k的dim维度上全部乘以sin,然后相加对应公式f_k(x_m, m)
    return q_embed, k_embed
        
